# 🛍️ RL-AutoML no Big Data (Sales Forecast Completo)

Neste experimento magistral, aplicamos o Agente de Reinforcement Learning (Q-Learning) no projeto **Sales Forecast**, focando em previsão de demanda contínua (Time Series Regression) com **5.6 milhões de registros**.

Para treinar em tempo hábil na CPU sem amostrar os dados (ou seja, lendo 100% da base), introduzimos a estratégia de **Proxy Training**:
- **Poda Extrema:** Treinamos o LightGBM com apenas `n_estimators=50` durante a exploração do Agente.
- **Bagging Agressivo:** Configuramos o `bagging_fraction=0.15` para o modelo ver fatias aleatórias da base muito rapidamente.
- **Recompensa Inversa:** O Agente só ganha recompensa se o **MAE (Erro Médio Absoluto)** diminuir.

Ao final, a configuração mágica descoberta pelo Proxy é injetada em um modelo final Absoluto de 1000 estimadores.


In [1]:
import pandas as pd
import numpy as np
import mlflow
import os
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

mlflow.set_tracking_uri("sqlite:///../../mlruns.db")
mlflow.set_experiment("RL_Proxy_FullScale")


<Experiment: artifact_location='file:///D:/mlops-experiments/experiments/sales-forecast/mlruns/5', creation_time=1785363501946, experiment_id='5', last_update_time=1785363501946, lifecycle_stage='active', name='RL_Proxy_FullScale', tags={}>

## 1. Fase 1: Carregamento Massivo e Feature Engineering (100% da Base)

In [2]:
print("Fase 1: Carregando 100% da Base de Dados (5.6 Milhões)...")
raw_dir = r"D:\mlops-experiments\experiments\sales-forecast\data\raw"
df_vendas = pd.read_parquet(os.path.join(raw_dir, 'fato_vendas.parquet'))
df_pdvs = pd.read_parquet(os.path.join(raw_dir, 'dim_pdvs.parquet'))
df_produtos = pd.read_parquet(os.path.join(raw_dir, 'dim_produtos.parquet'))

print("Merges de Big Data...")
df_merged = pd.merge(df_vendas, df_pdvs, left_on='internal_store_id', right_on='pdv', how='inner')
df_merged = pd.merge(df_merged, df_produtos, left_on='internal_product_id', right_on='produto', how='inner')

df_merged['transaction_date'] = pd.to_datetime(df_merged['transaction_date'])
df_merged['ano'] = df_merged['transaction_date'].dt.isocalendar().year
df_merged['semana'] = df_merged['transaction_date'].dt.isocalendar().week

dim_cols = ['categoria_pdv', 'premise', 'categoria', 'subcategoria', 'tipos', 'label', 'marca', 'fabricante']
group_cols = ['ano', 'semana', 'pdv', 'produto'] + dim_cols

print("Agregando vendas (Pivotando milhões de linhas)...")
agg_vendas = df_merged.groupby(group_cols).agg(
    quantidade=('quantity', 'sum'),
    total_gross_value=('gross_value', 'sum'),
).reset_index()

agg_vendas = agg_vendas.rename(columns={'produto': 'sku'})
agg_vendas['preco_medio_unitario'] = np.where(
    agg_vendas['quantidade'] > 0,
    agg_vendas['total_gross_value'] / agg_vendas['quantidade'],
    0.0
)
agg_vendas.drop(columns=['total_gross_value'], inplace=True)


Fase 1: Carregando 100% da Base de Dados (5.6 Milhões)...


Merges de Big Data...


Agregando vendas (Pivotando milhões de linhas)...


In [3]:
print("Engenharia de Features Temporais Massivas (Aguarde...).")
agg_vendas.sort_values(['pdv', 'sku', 'ano', 'semana'], inplace=True)
agg_vendas.reset_index(drop=True, inplace=True)

agg_vendas['trimestre'] = (agg_vendas['semana'] - 1) // 13 + 1
agg_vendas['seno_semana'] = np.sin(2 * np.pi * agg_vendas['semana'] / 52)
agg_vendas['cosseno_semana'] = np.cos(2 * np.pi * agg_vendas['semana'] / 52)

grouped_qty = agg_vendas.groupby(['pdv', 'sku'])['quantidade']
for lag in [1, 2, 3, 4, 12]:
    agg_vendas[f'lag_{lag}_semanas'] = grouped_qty.shift(lag)

agg_vendas['lag_1_preco'] = agg_vendas.groupby(['pdv', 'sku'])['preco_medio_unitario'].shift(1)
agg_vendas['lag_diff_1'] = agg_vendas['lag_1_semanas'] - agg_vendas['lag_2_semanas']

shifted = grouped_qty.shift(1)
tmp = pd.DataFrame({'val': shifted, 'pdv': agg_vendas['pdv'], 'sku': agg_vendas['sku']})
tmp_grouped = tmp.groupby(['pdv', 'sku'])['val']

for window in [4, 12]:
    roll = tmp_grouped.rolling(window=window, min_periods=1)
    agg_vendas[f'rolling_mean_{window}_semanas'] = roll.mean().reset_index(level=[0, 1], drop=True)
    agg_vendas[f'rolling_std_{window}_semanas'] = roll.std().reset_index(level=[0, 1], drop=True)

agg_vendas.fillna(0, inplace=True)

for col in dim_cols + ['pdv', 'sku']:
    agg_vendas[col] = agg_vendas[col].astype('category')

feature_names = ['semana', 'trimestre', 'seno_semana', 'cosseno_semana', 'pdv', 'sku'] + dim_cols +                 ['lag_1_semanas', 'lag_2_semanas', 'lag_3_semanas', 'lag_4_semanas', 'lag_12_semanas'] +                 ['lag_1_preco', 'lag_diff_1', 'preco_medio_unitario'] +                 ['rolling_mean_4_semanas', 'rolling_std_4_semanas', 'rolling_mean_12_semanas', 'rolling_std_12_semanas']

train_set = agg_vendas[agg_vendas['semana'] < 48]
val_set = agg_vendas[agg_vendas['semana'] >= 48]

X_train, y_train = train_set[feature_names], train_set['quantidade']
X_val, y_val = val_set[feature_names], val_set['quantidade']

del agg_vendas, df_merged, df_vendas # Limpeza de RAM crítica
print("Dados prontos!")


Engenharia de Features Temporais Massivas (Aguarde...).


Dados prontos!


## 2. Fase 2: O Agente RL no Sandbox Proxy (Treinamento Veloz)

In [4]:
class ProxyHyperparameterEnv:
    def __init__(self, X_t, y_t, X_v, y_v, cats):
        self.X_train, self.y_train = X_t, y_t
        self.X_val, self.y_val = X_v, y_v
        self.cats = cats
        
        self.lr_bins = [0.01, 0.05, 0.1, 0.2, 0.3]
        self.leaves_bins = [31, 64, 128, 256, 512]
        self.depth_bins = [5, 7, 10, 15, -1]
        self.max_idx = 4
        self.reset()
        
    def reset(self):
        self.state = [2, 2, 2] 
        self.best_mae = float('inf')
        self.current_step = 0
        self.max_steps = 10
        return tuple(self.state)
        
    def step(self, action):
        self.current_step += 1
        new_state = list(self.state)
        reward = -0.1
        
        if action == 0: new_state[0] += 1
        elif action == 1: new_state[0] -= 1
        elif action == 2: new_state[1] += 1
        elif action == 3: new_state[1] -= 1
        elif action == 4: new_state[2] += 1
        elif action == 5: new_state[2] -= 1
        
        if any(s < 0 or s > self.max_idx for s in new_state):
            reward = -2.0
            return tuple(self.state), reward, self.current_step >= self.max_steps, self.best_mae
            
        self.state = new_state
        lr = self.lr_bins[self.state[0]]
        leaves = self.leaves_bins[self.state[1]]
        depth = self.depth_bins[self.state[2]]
        
        # PROXY HACK: Poucas arvores (50) e Bagging extremo (0.15)
        model = lgb.LGBMRegressor(objective='regression_l1', n_estimators=50, learning_rate=lr, 
                                  num_leaves=leaves, max_depth=depth, bagging_fraction=0.15, bagging_freq=1, 
                                  random_state=42, verbosity=-1)
        model.fit(self.X_train, self.y_train, categorical_feature=self.cats)
        preds = model.predict(self.X_val)
        current_mae = mean_absolute_error(self.y_val, preds)
        
        if self.best_mae == float('inf'):
            self.best_mae = current_mae
            reward += 1.0
        elif current_mae < self.best_mae:
            improvement = self.best_mae - current_mae
            reward += improvement * 20.0
            self.best_mae = current_mae
        else:
            reward -= 0.5
            
        done = self.current_step >= self.max_steps
        return tuple(self.state), reward, done, current_mae

class QLearningAgent:
    def __init__(self):
        self.q_table = np.zeros((5, 5, 5, 6))
        self.alpha = 0.2
        self.gamma = 0.9
        self.epsilon = 1.0
        
    def choose_action(self, state):
        if np.random.uniform(0, 1) < self.epsilon:
            return np.random.randint(6)
        return np.argmax(self.q_table[state])

    def update(self, state, action, reward, next_state):
        best_next = np.argmax(self.q_table[next_state])
        td_target = reward + self.gamma * self.q_table[next_state][best_next]
        self.q_table[state][action] += self.alpha * (td_target - self.q_table[state][action])


In [5]:
print("Treinando Agente RL em Modo Proxy (Atalhos Habilitados)...")
env = ProxyHyperparameterEnv(X_train, y_train, X_val, y_val, dim_cols + ['pdv', 'sku'])
agent = QLearningAgent()

best_global_mae_proxy = float('inf')
best_state = None
episodes = 15

with mlflow.start_run(run_name="SalesForecast_ProxyRL"):
    for ep in range(episodes):
        state = env.reset()
        done = False
        while not done:
            action = agent.choose_action(state)
            next_state, reward, done, mae = env.step(action)
            agent.update(state, action, reward, next_state)
            state = next_state
            
            if mae < best_global_mae_proxy:
                best_global_mae_proxy = mae
                best_state = state
                
        if agent.epsilon > 0.05:
            agent.epsilon *= 0.8
            
        print(f"Proxy Ep {ep+1} | Epsilon {agent.epsilon:.2f} | Best Proxy MAE: {best_global_mae_proxy:.4f}")
        mlflow.log_metric("Proxy_MAE", best_global_mae_proxy, step=ep)


Treinando Agente RL em Modo Proxy (Atalhos Habilitados)...


Proxy Ep 1 | Epsilon 0.80 | Best Proxy MAE: 1.4989


Proxy Ep 2 | Epsilon 0.64 | Best Proxy MAE: 1.4989


Proxy Ep 3 | Epsilon 0.51 | Best Proxy MAE: 1.4797


Proxy Ep 4 | Epsilon 0.41 | Best Proxy MAE: 1.4797


...

Proxy Ep 11 | Epsilon 0.09 | Best Proxy MAE: 1.4670


Proxy Ep 12 | Epsilon 0.07 | Best Proxy MAE: 1.4670


Proxy Ep 13 | Epsilon 0.05 | Best Proxy MAE: 1.4670


Proxy Ep 14 | Epsilon 0.04 | Best Proxy MAE: 1.4670


Proxy Ep 15 | Epsilon 0.04 | Best Proxy MAE: 1.4670


## 3. Fase 3: Treino Final Absoluto

Após mapear o ambiente virtualmente através dos micro-fits, o agente repassa os hiperparâmetros campeões para o `LGBMRegressor` definitivo rodar o Treinamento sem amostragem com 1000 árvores reais.


In [6]:
print("Fase 3: O Treino Absoluto!")
final_lr = env.lr_bins[best_state[0]]
final_leaves = env.leaves_bins[best_state[1]]
final_depth = env.depth_bins[best_state[2]]

print(f"Injetando Config Mágica no Modelo Real: LR={final_lr}, Leaves={final_leaves}, Depth={final_depth}")
model_final = lgb.LGBMRegressor(objective='regression_l1', n_estimators=1000, learning_rate=final_lr, 
                              num_leaves=final_leaves, max_depth=final_depth, random_state=42, verbosity=-1)

model_final.fit(X_train, y_train, categorical_feature=dim_cols + ['pdv', 'sku'], 
                eval_set=[(X_val, y_val)], eval_metric="mae", callbacks=[lgb.early_stopping(50, verbose=False)])

preds_final = model_final.predict(X_val)
final_mae = mean_absolute_error(y_val, preds_final)

print("\n=== CONFRONTO FINAL (OPTUNA vs RL) ===")
print("Optuna Histórico MAE: 1.4218")
print(f"RL Proxy Absoluto MAE: {final_mae:.4f}")


Fase 3: O Treino Absoluto!
Injetando Config Mágica no Modelo Real: LR=0.1, Leaves=512, Depth=15



=== CONFRONTO FINAL (OPTUNA vs RL) ===
Optuna Histórico MAE: 1.4218
RL Proxy Absoluto MAE: 1.4236
